# 🚀 NEXinfra AI — Google Colab YOLOv8 Model Trainer
### Fine-Tune YOLO on 6 Municipal Defect Classes with Free GPU & Export to ONNX

This notebook trains an Ultralytics YOLOv8 model on the 6 canonical civic infrastructure defect categories and exports `model.onnx` for the NEXinfra Central AI Engine:
1. `0: pothole_road_defect` (Road Damage / Pothole)
2. `1: water_drainage_burst` (Water / Drainage Burst)
3. `2: garbage_waste_overflow` (Solid Waste Overflow)
4. `3: electrical_hazard` (Electrical & Streetlight)
5. `4: structural_bridge_crack` (Structural Anomaly / Bridge Crack)
6. `5: tree_greenery_hazard` (Public Park & Greenery Hazard)

In [ ]:
# Step 1: Install Ultralytics & ONNX Dependencies
!pip install -q ultralytics onnx onnxslim onnxruntime
import torch
print(f"✅ GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"⚡ GPU Device: {torch.cuda.get_device_name(0)}")

In [ ]:
# Step 2: Upload or Create Dataset
# You can upload 'nexinfra_dataset.zip' directly to Colab files, or this cell will unzip it
import os
import zipfile
import yaml

if os.path.exists('nexinfra_dataset.zip'):
    with zipfile.ZipFile('nexinfra_dataset.zip', 'r') as zip_ref:
        zip_ref.extractall('nexinfra_data')
    print("✅ Unzipped nexinfra_dataset.zip")
else:
    print("ℹ️ Upload 'nexinfra_dataset.zip' using the Colab left file panel")

# Write Dataset YAML
dataset_config = {
    'path': '/content/nexinfra_data/dataset' if os.path.exists('/content/nexinfra_data/dataset') else './dataset',
    'train': 'images/train',
    'val': 'images/val',
    'nc': 6,
    'names': {
        0: 'pothole_road_defect',
        1: 'water_drainage_burst',
        2: 'garbage_waste_overflow',
        3: 'electrical_hazard',
        4: 'structural_bridge_crack',
        5: 'tree_greenery_hazard'
    }
}

with open('civic_defects.yaml', 'w') as f:
    yaml.dump(dataset_config, f, default_flow_style=False)

print("✅ Generated civic_defects.yaml")

In [ ]:
# Step 3: Train YOLOv8 on GPU (50 Epochs)
from ultralytics import YOLO

# Load pretrained YOLOv8n or YOLOv8s
model = YOLO('yolov8n.pt')

results = model.train(
    data='civic_defects.yaml',
    epochs=50,
    imgsz=640,
    batch=16,
    device=0 if torch.cuda.is_available() else 'cpu',
    name='nexinfra_colab_run',
    save=True,
    plots=True,
    augment=True
)

In [ ]:
# Step 4: Export to Optimized ONNX Format
best_weights = 'runs/detect/nexinfra_colab_run/weights/best.pt'
trained_model = YOLO(best_weights)

# Export to ONNX (opset 17 with onnxslim)
onnx_path = trained_model.export(format='onnx', imgsz=640, opset=17, simplify=True)
print(f"🎉 Exported ONNX to: {onnx_path}")

In [ ]:
# Step 5: Download trained ONNX & PyTorch weights to your PC
from google.colab import files
import os

onnx_file = 'runs/detect/nexinfra_colab_run/weights/best.onnx'
if os.path.exists(onnx_file):
    # Rename to model.onnx for easy drop-in
    os.rename(onnx_file, 'model.onnx')
    print("📥 Downloading model.onnx...")
    files.download('model.onnx')
    files.download(best_weights)
else:
    print("❌ ONNX file not found. Check training logs above.")